# Notebook 06: Training — JAM Condition

**Agency Calculus Empirical Validation — Paper C**

Social planner objective: **R = log(min(u_i))**

**NO EPSILON.** The singularity at min_u = 0 must be preserved.

This is the Agency Calculus JAM objective. The gradient of log(x) as x→0+
is +∞, which means the planner faces an infinite cost for driving any agent
to zero utility. This is the structural guarantee that prevents floor compression.

**Predicted outcomes:**
- Floor utility: rises over training (the planner actively protects the floor)
- Gini coefficient: lowest of three conditions
- Tax on floor agent: minimal (planner cannot extract from floor)
- Total utility: lower than SUM/NASH but floor utility is highest

The critical result: **JAM produces lower total utility but higher floor utility.**
This IS the compensation mechanism. Under SUM, the planner sacrifices the floor
for aggregate gain. Under JAM, it cannot.

**Training stability note:** If gradients are unstable near zero, increase
batch size or lower learning rate. Do NOT add epsilon to the JAM reward.

5 seeds × 10M steps each.\n\nImportant: this notebook assumes your AI Economist fork already applies the `jam` planner objective inside the real env/scenario reward path. The local `training.py` callback only logs diagnostics.

In [ ]:
# ── Environment check ─────────────────────────────────────────────────────
# If ai_economist is missing, run notebook 01 first (it handles installation
# and the required kernel restart).
import sys, os

try:
    import ai_economist  # noqa: F401
except ModuleNotFoundError:
    raise SystemExit(
        "\n❌  ai_economist not found. Run notebook 01_setup_and_test first,\n"
        "    restart the kernel, then return here."
    )

# Add src/ to path
for candidate in [
    '/content/ac-validation/src',
    os.path.join(os.getcwd(), '..', 'src'),
    os.path.join(os.getcwd(), 'src'),
]:
    if os.path.exists(candidate) and candidate not in sys.path:
        sys.path.insert(0, candidate)
        print(f'src on path: {candidate}')
        break


In [ ]:
import sys, os
import numpy as np
import math

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from training import run_training, TOTAL_TIMESTEPS, N_SEEDS
from ac_rewards import jam_reward
from metrics import MetricsLogger
import matplotlib.pyplot as plt

CONDITION = 'jam'
RESULTS_DIR = '../results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Condition: {CONDITION}  (R = log(min(u_i)), NO EPSILON)')
print('Fork requirement: planner reward must be swapped in AI Economist, not only logged in training.py')

# Verify JAM reward function
test_utils = [10.0, 5.0, 8.0, 2.0]
expected = math.log(2.0)
actual = jam_reward(test_utils)
assert abs(actual - expected) < 1e-10, f'JAM reward mismatch: {actual} != {expected}'
print(f'JAM reward check: log({min(test_utils)}) = {actual:.6f} ✓')

# Verify zero guard
assert jam_reward([0.0, 5.0, 8.0, 2.0]) == -1e10, 'Zero guard failed!'
print('Zero guard: -1e10 ✓ (not epsilon stabilization)')

## Training Stability Considerations

JAM's infinite gradient near zero can cause instability if the floor agent
ever reaches zero utility. This is expected behavior — the gradient is doing
its job. If instability occurs:

1. **Increase batch size**: More samples per update = smoother gradient estimates
2. **Lower learning rate**: `lr = 1e-4` instead of `3e-4`
3. **Clip gradients**: `grad_clip = 40.0` (RLlib default is 40)
4. **Do NOT add epsilon to JAM reward** — that would destroy the experiment

The -1e10 floor in the code signals a training failure, not a design choice.

In [ ]:
# Run a single seed (set SEED = 0, 1, 2, 3, 4 across separate sessions)
SEED = 0  # Change this for each session

save_path = f'{RESULTS_DIR}/{CONDITION}_seed{SEED}_metrics.npz'
if os.path.exists(save_path):
    print(f'Seed {SEED} already complete: {save_path}')
else:
    print(f'Starting training: condition={CONDITION}, seed={SEED}')
    print('CRITICAL: Using JAM reward with NO epsilon')
    logger = run_training(
        condition=CONDITION,
        seed=SEED,
        total_timesteps=TOTAL_TIMESTEPS,
        results_dir=RESULTS_DIR,
    )
    print(f'Training complete. Saved to {save_path}')

In [ ]:
# DEBUG: short Colab smoke run
# logger_debug = run_training(
#     condition=CONDITION,
#     seed=99,
#     total_timesteps=20_000,
#     eval_interval=5_000,
#     results_dir=RESULTS_DIR,
# )
print('Debug run commented out. Recommended first Colab smoke test: 20K steps, eval every 5K.')

In [ ]:
def plot_seed_progress(condition, seed, results_dir=RESULTS_DIR):
    path = f'{results_dir}/{condition}_seed{seed}_metrics.npz'
    if not os.path.exists(path):
        return
    logger = MetricsLogger.load(path)
    arrays = logger.to_arrays()
    steps = arrays.get('step', np.array([]))
    metrics_to_plot = ['floor_utility_mean', 'total_utility_mean', 'gini_wealth_mean']
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, metric in zip(axes, metrics_to_plot):
        if metric in arrays:
            ax.plot(steps, arrays[metric], color='#2ecc71', linewidth=2)
            ax.set_title(metric)
            ax.set_xlabel('Training Steps')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
    plt.suptitle(f'JAM Condition — Seed {seed}', y=1.02)
    plt.tight_layout()
    plt.show()

for s in range(N_SEEDS):
    plot_seed_progress(CONDITION, s)

## Monitoring the Singularity

Track how often the -1e10 floor is hit during training.
If it's being hit frequently, the training setup needs adjustment
(not the reward function).

In [ ]:
# Load completed seeds and check floor_utility trajectory
# The key question: does floor_utility rise over time?

completed_seeds = []
for s in range(N_SEEDS):
    path = f'{RESULTS_DIR}/{CONDITION}_seed{s}_metrics.npz'
    if os.path.exists(path):
        completed_seeds.append(s)

if completed_seeds:
    print(f'Completed seeds: {completed_seeds}')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    for s in completed_seeds:
        logger = MetricsLogger.load(f'{RESULTS_DIR}/{CONDITION}_seed{s}_metrics.npz')
        arrays = logger.to_arrays()
        steps = arrays.get('step', np.array([]))
        if 'floor_utility_mean' in arrays:
            ax1.plot(steps, arrays['floor_utility_mean'],
                     alpha=0.7, label=f'seed {s}')
        if 'total_utility_mean' in arrays:
            ax2.plot(steps, arrays['total_utility_mean'],
                     alpha=0.7, label=f'seed {s}')
    
    ax1.set_title('JAM: Floor Utility (should rise)')
    ax1.set_xlabel('Steps')
    ax1.legend()
    ax2.set_title('JAM: Total Utility (lower than SUM, stable)')
    ax2.set_xlabel('Steps')
    ax2.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No completed seeds yet. Run training first.')

## Notes on Interpreting Results

**If JAM floor utility rises → prediction confirmed.** The structural
guarantee works: infinite gradient prevents floor compression.

**If JAM floor utility is similar to NASH:** Check that epsilon was NOT
added to the reward function. Also verify the fork-side planner reward swap
is correctly replacing the default AI Economist reward.

**If training diverges (NaN rewards):** Lower lr to 1e-4, increase
batch_size to 8000. The singularity is the cause, not a bug.

**Negative result note:** If JAM does not outperform NASH on floor utility
in this RL setting, that is still a scientifically valid result. Report
the exact comparison honestly.

**Next:** Notebook 07 — cross-condition analysis and plots.